In [ ]:
# ============================================================
# ÉTAPE 2 : FEATURE ENGINEERING — VERSION FINALE COMPLÈTE
# ============================================================
#
# Structure :
#   A. Chargement et conversions de types
#   B. Historique long format (une ligne par joueur par match)
#   C. Rolling features (forme récente, stats de service)
#   D. Features H2H (historique tête-à-tête)
#   E. Assemblage final symétrique (vue P1 vs P2)
#   F. Nettoyage + nouvelles features dérivées
#   G. Sauvegarde → atp_features_clean_final.parquet + .csv
#
# ⚡ Version vectorisée : pas d'iterrows()
# 🔒 No data leakage : shift(1) sur tous les rolling/cumsum
# ============================================================

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# A. CHARGEMENT ET CONVERSIONS DE TYPES
# ─────────────────────────────────────────────────────────────

df = pd.read_csv("../../data/tennis/atp_clean.csv", parse_dates=["tourney_date"])
df = df.sort_values("tourney_date").reset_index(drop=True)

for col in ["winner_rank","loser_rank","winner_rank_points","loser_rank_points",
            "winner_ht","loser_ht","winner_age","loser_age",
            "w_ace","w_df","w_svpt","w_1stIn","w_1stWon","w_2ndWon","w_bpSaved","w_bpFaced",
            "l_ace","l_df","l_svpt","l_1stIn","l_1stWon","l_2ndWon","l_bpSaved","l_bpFaced",
            "minutes"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["winner_rank"]        = df["winner_rank"].fillna(999)
df["loser_rank"]         = df["loser_rank"].fillna(999)
df["winner_rank_points"] = df["winner_rank_points"].fillna(0)
df["loser_rank_points"]  = df["loser_rank_points"].fillna(0)

print(f"✅ Chargé : {len(df):,} matchs")


# ─────────────────────────────────────────────────────────────
# B. HISTORIQUE LONG FORMAT
# ─────────────────────────────────────────────────────────────

def safe_div(a, b):
    return np.where(b > 0, a / b, np.nan)

w = pd.DataFrame({
    "player_id"    : df["winner_id"],
    "date"         : df["tourney_date"],
    "surface"      : df["surface"],
    "won"          : 1,
    "rank"         : df["winner_rank"],
    "rank_points"  : df["winner_rank_points"],
    "age"          : df["winner_age"],
    "ht"           : df["winner_ht"],
    "ace_rate"     : safe_div(df["w_ace"],     df["w_svpt"]),
    "df_rate"      : safe_div(df["w_df"],      df["w_svpt"]),
    "fs_in_pct"    : safe_div(df["w_1stIn"],   df["w_svpt"]),
    "fs_won_pct"   : safe_div(df["w_1stWon"],  df["w_1stIn"]),
    "ss_won_pct"   : safe_div(df["w_2ndWon"],  df["w_svpt"] - df["w_1stIn"]),
    "bp_saved_pct" : safe_div(df["w_bpSaved"], df["w_bpFaced"]),
    "match_ref"    : df.index,
    "is_winner"    : True,
})

l = pd.DataFrame({
    "player_id"    : df["loser_id"],
    "date"         : df["tourney_date"],
    "surface"      : df["surface"],
    "won"          : 0,
    "rank"         : df["loser_rank"],
    "rank_points"  : df["loser_rank_points"],
    "age"          : df["loser_age"],
    "ht"           : df["loser_ht"],
    "ace_rate"     : safe_div(df["l_ace"],     df["l_svpt"]),
    "df_rate"      : safe_div(df["l_df"],      df["l_svpt"]),
    "fs_in_pct"    : safe_div(df["l_1stIn"],   df["l_svpt"]),
    "fs_won_pct"   : safe_div(df["l_1stWon"],  df["l_1stIn"]),
    "ss_won_pct"   : safe_div(df["l_2ndWon"],  df["l_svpt"] - df["l_1stIn"]),
    "bp_saved_pct" : safe_div(df["l_bpSaved"], df["l_bpFaced"]),
    "match_ref"    : df.index,
    "is_winner"    : False,
})

history = pd.concat([w, l], ignore_index=True)
history = history.sort_values(["player_id", "date", "match_ref"]).reset_index(drop=True)
print(f"✅ Historique : {len(history):,} entrées")


# ─────────────────────────────────────────────────────────────
# C. ROLLING FEATURES
# ─────────────────────────────────────────────────────────────

WINDOWS   = [5, 10, 20]
STAT_COLS = ["won", "ace_rate", "df_rate", "fs_in_pct",
             "fs_won_pct", "ss_won_pct", "bp_saved_pct"]

rolling_parts = [history[["player_id", "date", "surface", "match_ref", "is_winner"]].copy()]

for W in WINDOWS:
    print(f"  Rolling W={W}...", end=" ")
    grp = history.groupby("player_id")
    for col in STAT_COLS:
        rolling_parts.append(
            grp[col]
            .transform(lambda x: x.shift(1).rolling(W, min_periods=1).mean())
            .rename(f"last{W}_{col}")
        )
    rolling_parts.append(
        grp["won"]
        .transform(lambda x: x.shift(1).rolling(W, min_periods=1).count())
        .rename(f"last{W}_n_matches")
    )
    print("✅")

for surf_val in history["surface"].dropna().unique():
    col_name = f"won_surf_{surf_val}"
    history[col_name] = np.where(history["surface"] == surf_val, history["won"], np.nan)

for W in WINDOWS:
    for surf_val in history["surface"].dropna().unique():
        col_name   = f"won_surf_{surf_val}"
        result_col = f"last{W}_win_rate_{surf_val}"
        rolling_parts.append(
            history.groupby("player_id")[col_name]
            .transform(lambda x: x.shift(1).rolling(W, min_periods=1).mean())
            .rename(result_col)
        )

print("  Fatigue 14d...", end=" ")
fatigue_series = pd.Series(index=history.index, dtype=float)
for pid, grp in history.groupby("player_id"):
    grp_sorted = grp.sort_values(["date", "match_ref"])
    fat = (
        grp_sorted.set_index("date")["won"]
        .shift(1)
        .rolling("14D", min_periods=0)
        .count()
    )
    fatigue_series.loc[grp_sorted.index] = fat.values
history["fatigue_14d"] = fatigue_series
rolling_parts.append(history["fatigue_14d"])

print("✅\n  Days since last match...", end=" ")
history["days_since_last"] = (
    history.groupby("player_id")["date"]
    .transform(lambda x: x.diff().dt.days)
)
rolling_parts.append(history["days_since_last"])
print("✅")

history_feat = pd.concat(rolling_parts, axis=1)
history_feat = history_feat.loc[:, ~history_feat.columns.duplicated()]
print(f"✅ Rolling done : {history_feat.shape}")


# ─────────────────────────────────────────────────────────────
# D. FEATURES H2H
# ─────────────────────────────────────────────────────────────

print("  H2H (vectorisé)...", end=" ")

df["winner_id"] = df["winner_id"].astype(str).str.strip()
df["loser_id"]  = df["loser_id"].astype(str).str.strip()

df["p_lo"] = df[["winner_id", "loser_id"]].min(axis=1)
df["p_hi"] = df[["winner_id", "loser_id"]].max(axis=1)
df["pair"] = df["p_lo"] + "_" + df["p_hi"]
df["winner_is_lo"] = (df["winner_id"] == df["p_lo"]).astype(int)

df["h2h_lo_wins"] = df.groupby("pair")["winner_is_lo"].transform(
    lambda x: x.shift(1).cumsum().fillna(0)
)
df["h2h_n"] = df.groupby("pair")["winner_is_lo"].transform(
    lambda x: x.shift(1).expanding().count().fillna(0)
)
df["h2h_hi_wins"] = df["h2h_n"] - df["h2h_lo_wins"]

df["h2h_win_rate_winner"] = np.where(
    df["h2h_n"] > 0,
    np.where(df["winner_id"] == df["p_lo"],
             df["h2h_lo_wins"],
             df["h2h_hi_wins"]) / df["h2h_n"],
    np.nan
)

df["surf_match"] = df["surface"]
df_h2h_surf = (
    df.assign(w_is_lo=df["winner_is_lo"])
      .groupby(["pair", "surface"])
      .apply(lambda g: pd.Series({
          "idx"           : g.index.tolist(),
          "lo_wins_cumul" : g["winner_is_lo"].shift(1).cumsum().fillna(0).tolist(),
          "n_cumul"       : g["winner_is_lo"].shift(1).expanding().count().fillna(0).tolist(),
      }))
)

h2h_surf_rows = []
for (pair, surf), row in df_h2h_surf.iterrows():
    for idx, lo_w, n in zip(row["idx"], row["lo_wins_cumul"], row["n_cumul"]):
        h2h_surf_rows.append({"orig_idx": idx, "h2h_surf_lo_wins": lo_w, "h2h_surf_n": n})

h2h_surf_df = pd.DataFrame(h2h_surf_rows).set_index("orig_idx")
df = df.join(h2h_surf_df)

df["h2h_win_rate_surf_winner"] = np.where(
    df["h2h_surf_n"] > 0,
    np.where(df["winner_id"] == df["p_lo"],
             df["h2h_surf_lo_wins"],
             df["h2h_surf_n"] - df["h2h_surf_lo_wins"]) / df["h2h_surf_n"],
    np.nan
)
print("✅")


# ─────────────────────────────────────────────────────────────
# E. ASSEMBLAGE FINAL SYMÉTRIQUE
# ─────────────────────────────────────────────────────────────

history_feat_indexed       = history_feat.copy()
history_feat_indexed.index = history_feat["match_ref"]

winner_feat = history_feat_indexed[history_feat_indexed["is_winner"] == True]
loser_feat  = history_feat_indexed[history_feat_indexed["is_winner"] == False]

feat_cols = [c for c in history_feat.columns
             if c not in ["player_id","date","surface","match_ref","is_winner"]]

round_order  = {"R128":1,"R64":2,"R32":3,"R16":4,"QF":5,"SF":6,"F":7,"RR":3,"BR":6}
level_order  = {"G":5,"M":4,"F":4,"A":3,"D":2,"C":1}
hand_map     = {"R":1,"L":-1,"U":0}
indoor_map   = {"Y":1,"N":0,"Unknown":0}

df["round_num"]         = df["round"].map(round_order).fillna(3).astype(int)
df["tourney_level_num"] = df["tourney_level"].map(level_order).fillna(2).astype(int)
df["winner_hand_enc"]   = df["winner_hand"].map(hand_map).fillna(0).astype(int)
df["loser_hand_enc"]    = df["loser_hand"].map(hand_map).fillna(0).astype(int)
df["indoor_enc"]        = df["indoor"].map(indoor_map).fillna(0).astype(int)
df["month"]             = df["tourney_date"].dt.month

surf_dummies = pd.get_dummies(df["surface"], prefix="surf", dtype=int)
df = pd.concat([df, surf_dummies], axis=1)
surf_cols = surf_dummies.columns.tolist()
ctx_cols  = ["round_num","tourney_level_num","indoor_enc","month","best_of"] + surf_cols

def build_row(match_idx, p1_is_winner: bool):
    row  = df.loc[match_idx]
    p1_f = winner_feat.loc[match_idx] if p1_is_winner else loser_feat.loc[match_idx]
    p2_f = loser_feat.loc[match_idx]  if p1_is_winner else winner_feat.loc[match_idx]

    out  = {}
    pfx1 = "winner" if p1_is_winner else "loser"
    pfx2 = "loser"  if p1_is_winner else "winner"

    out["p1_rank"]     = row[f"{pfx1}_rank"]
    out["p1_rank_pts"] = row[f"{pfx1}_rank_points"]
    out["p1_seed"]     = row[f"{pfx1}_seed"]
    out["p1_age"]      = row[f"{pfx1}_age"]
    out["p1_ht"]       = row[f"{pfx1}_ht"]
    out["p1_hand"]     = row[f"{pfx1}_hand_enc"]
    out["p2_rank"]     = row[f"{pfx2}_rank"]
    out["p2_rank_pts"] = row[f"{pfx2}_rank_points"]
    out["p2_seed"]     = row[f"{pfx2}_seed"]
    out["p2_age"]      = row[f"{pfx2}_age"]
    out["p2_ht"]       = row[f"{pfx2}_ht"]
    out["p2_hand"]     = row[f"{pfx2}_hand_enc"]

    out["diff_rank"]     = out["p1_rank"]     - out["p2_rank"]
    out["diff_rank_pts"] = out["p1_rank_pts"] - out["p2_rank_pts"]
    out["diff_seed"]     = out["p1_seed"]     - out["p2_seed"]
    out["diff_age"]      = out["p1_age"]      - out["p2_age"]
    out["diff_ht"]       = out["p1_ht"]       - out["p2_ht"]
    out["same_hand"]     = int(out["p1_hand"] == out["p2_hand"])

    for col in feat_cols:
        v1 = p1_f[col]
        v2 = p2_f[col]
        out[f"p1_{col}"] = v1
        out[f"p2_{col}"] = v2
        if pd.api.types.is_number(v1) and pd.api.types.is_number(v2):
            out[f"diff_{col}"] = v1 - v2

    h2h_wr = row["h2h_win_rate_winner"]
    h2h_s  = row["h2h_win_rate_surf_winner"]
    out["h2h_n"]             = row["h2h_n"]
    out["h2h_win_rate_p1"]   = h2h_wr if p1_is_winner else (1 - h2h_wr if not np.isnan(h2h_wr) else np.nan)
    out["h2h_win_rate_surf"] = h2h_s  if p1_is_winner else (1 - h2h_s  if not np.isnan(h2h_s)  else np.nan)

    for col in ctx_cols:
        out[col] = row[col]

    out["p1_name"] = row["winner_name"] if p1_is_winner else row["loser_name"]
    out["p2_name"] = row["loser_name"]  if p1_is_winner else row["winner_name"]
    out["label"]   = int(p1_is_winner)
    return out

print("Assemblage final...", end=" ")
valid_indices = [i for i in df.index
                 if i in winner_feat.index and i in loser_feat.index]

skipped = len(df) - len(valid_indices)
if skipped > 0:
    print(f"\n⚠️  {skipped} matchs ignorés (features manquantes)")

all_rows = []
for i in valid_indices:
    all_rows.append(build_row(i, True))
    all_rows.append(build_row(i, False))

features_df = pd.DataFrame(all_rows)
print(f"✅  {len(features_df):,} lignes × {features_df.shape[1]} colonnes")
print(f"\nVérification symétrie :")
print(features_df["label"].value_counts().sort_index())

year_map           = df["tourney_date"].dt.year
features_df["year"] = [year_map[i] for i in valid_indices] * 2

print("\nVérification symétrie par année :")
print(features_df.groupby("year")["label"].value_counts().unstack())


# ─────────────────────────────────────────────────────────────
# F. NETTOYAGE + NOUVELLES FEATURES DÉRIVÉES
# ─────────────────────────────────────────────────────────────
# Toutes les features dérivées sont calculées ici pour que
# le notebook d'entraînement n'ait PAS à les recalculer.
# Il lui suffira de charger le parquet et splitter les données.

# ── F1. Suppression colonnes leakage et redondantes ──
cols_to_drop = [
    # Redondants
    "p1_rank_pts", "p2_rank_pts",
    "p1_seed", "p2_seed",
    "p1_hand", "p2_hand",
    # Fenêtres last5 brutes (garder seulement les diff)
    "p1_last5_ace_rate",    "p2_last5_ace_rate",
    "p1_last5_df_rate",     "p2_last5_df_rate",
    "p1_last5_fs_in_pct",   "p2_last5_fs_in_pct",
    "p1_last5_fs_won_pct",  "p2_last5_fs_won_pct",
    "p1_last5_ss_won_pct",  "p2_last5_ss_won_pct",
    "p1_last5_bp_saved_pct","p2_last5_bp_saved_pct",
    # Fenêtres last20 brutes
    "p1_last20_ace_rate",    "p2_last20_ace_rate",
    "p1_last20_df_rate",     "p2_last20_df_rate",
    "p1_last20_fs_in_pct",   "p2_last20_fs_in_pct",
    "p1_last20_fs_won_pct",  "p2_last20_fs_won_pct",
    "p1_last20_ss_won_pct",  "p2_last20_ss_won_pct",
    "p1_last20_bp_saved_pct","p2_last20_bp_saved_pct",
    # n_matches redondants
    "diff_last5_n_matches",  "p1_last5_n_matches",  "p2_last5_n_matches",
    "diff_last20_n_matches", "p1_last20_n_matches", "p2_last20_n_matches",
    # Win rate Grass : 83% NaN
    "p1_last5_win_rate_Grass",  "p2_last5_win_rate_Grass",  "diff_last5_win_rate_Grass",
    "p1_last10_win_rate_Grass", "p2_last10_win_rate_Grass", "diff_last10_win_rate_Grass",
    "p1_last20_win_rate_Grass", "p2_last20_win_rate_Grass", "diff_last20_win_rate_Grass",
    # Leakage symétrique sur fatigue
    "diff_fatigue_14d",
    "diff_days_since_last",
    # Corrélé 0.87 avec diff_last10_won
    "diff_last20_won",
]

cols_to_drop_existing = [c for c in cols_to_drop if c in features_df.columns]
cols_not_found        = [c for c in cols_to_drop if c not in features_df.columns]
features_df = features_df.drop(columns=cols_to_drop_existing)

print(f"\n✅ Colonnes supprimées : {len(cols_to_drop_existing)}")
if cols_not_found:
    print(f"⚠️  Introuvables (ignorées) : {cols_not_found}")
print(f"   Colonnes restantes : {features_df.shape[1]}")

# ── F2. Nouvelles features dérivées ──
# 1. Ratio de classement
features_df["rank_ratio"] = (
    features_df["p1_rank"] / features_df["p2_rank"].replace(0, np.nan)
).clip(0.01, 100)

# 2. Momentum : tendance de forme récente
features_df["p1_momentum"]  = features_df["p1_last5_won"] - features_df["p1_last20_won"]
features_df["p2_momentum"]  = features_df["p2_last5_won"] - features_df["p2_last20_won"]
features_df["diff_momentum"] = features_df["p1_momentum"] - features_df["p2_momentum"]

# 3. Service dominance
features_df["p1_serve_dom"] = (
    features_df["p1_last10_ace_rate"] +
    features_df["p1_last10_fs_won_pct"] -
    features_df["p1_last10_df_rate"]
)
features_df["p2_serve_dom"] = (
    features_df["p2_last10_ace_rate"] +
    features_df["p2_last10_fs_won_pct"] -
    features_df["p2_last10_df_rate"]
)
features_df["diff_serve_dom"] = features_df["p1_serve_dom"] - features_df["p2_serve_dom"]

# 4. Pression sous break
features_df["diff_bp_pressure"] = (
    features_df["diff_last10_bp_saved_pct"] -
    features_df["diff_last10_fs_won_pct"]
)

print(f"✅ Nouvelles features ajoutées : rank_ratio, momentum, serve_dom, bp_pressure")
print(f"   Colonnes totales : {features_df.shape[1]}")

# ── F3. Tri chronologique ──
features_df = features_df.sort_values("year").reset_index(drop=True)

# ── F4. Rapport NaN ──
X = features_df.drop(columns=["label","year","p1_name","p2_name"], errors="ignore")
y = features_df["label"]
nan_report = X.isnull().mean().mul(100).sort_values(ascending=False)
print("\nTop colonnes avec NaN (%) :")
print(nan_report[nan_report > 0].head(10).to_string())


# ─────────────────────────────────────────────────────────────
# G. SAUVEGARDE FINALE
# ─────────────────────────────────────────────────────────────

# Vérification noms joueurs
print("\nVérification p1_name/p2_name :")
print(features_df[["p1_name","p2_name","label","year"]].head(4).to_string())

# Sauvegarde brute (avant nettoyage) déjà faite à l'étape F
# Sauvegarde finale nettoyée
features_df.to_parquet("../../data/tennis/atp_features_clean_final.parquet", index=False)
features_df.to_csv("../../data/tennis/atp_features_clean_final.csv", index=False)

print(f"\n{'='*55}")
print(f"  ✅ Sauvegardé → atp_features_clean_final.parquet")
print(f"  ✅ Sauvegardé → atp_features_clean_final.csv")
print(f"  Features       : {X.shape[1]}")
print(f"  Lignes totales : {len(features_df):,}")
print(f"  Label 0/1      : {y.value_counts().to_dict()}")
print(f"{'='*55}")
print("\nRechargement dans le notebook d'entraînement :")
print("  df = pd.read_parquet('atp_features_clean_final.parquet')")
